# ปฏิบัติการที่ 6 การค้นหาแบบมีคู่แข่งขันและปัญหาที่มีข้อจำกัด: Minimax และ CSP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NKSR22/Introduction_to_AI_Labs/blob/main/lab06-adversarial-search-csp.ipynb)

ปฏิบัติการประกอบหนังสือ *ปัญญาประดิษฐ์เบื้องต้น* (นครินทร์ ศรีปัญญา) บทที่ 2 หัวข้อ 2.5 การค้นหาแบบมีคู่แข่งขัน และ หัวข้อ 2.6 ปัญหาที่มีข้อจำกัด

## วัตถุประสงค์

เมื่อทำปฏิบัติการนี้จบ ผู้เรียนสามารถ

1. คำนวณค่า Minimax บนต้นไม้เกมด้วยโค้ด และแสดงการตัดกิ่งด้วย Alpha-Beta Pruning ทีละขั้นได้
2. เขียน AI เล่นเกม OX (Tic-Tac-Toe) ด้วย Minimax แล้ววัดจำนวนโหนดที่ Alpha-Beta ประหยัดได้
3. ใช้ฟังก์ชันประเมิน (Evaluation Function) ร่วมกับการค้นหาแบบจำกัดความลึกได้
4. นิยามปัญหาในรูป CSP (ตัวแปร โดเมน ข้อจำกัด) และแก้ด้วย Backtracking Search ได้ ทั้งการจัดตารางซ่อมบำรุง การระบายสีแผนที่ และซูโดกุ
5. เพิ่ม MRV และ Forward Checking ให้ Backtracking แล้ววัดจำนวนการลองค่าและการย้อนรอยที่ลดลงได้

## เนื้อหาในหนังสือที่เกี่ยวข้อง

- 2.5.1 เกมและ Game Tree — ตัวอย่างที่ 2.22
- 2.5.2 Minimax — ตัวอย่างที่ 2.23 และ 2.24
- 2.5.3 Alpha-Beta Pruning — ตัวอย่างที่ 2.25
- 2.6.1 การนิยาม CSP — ตัวอย่างที่ 2.26
- 2.6.2 Backtracking Search — ตัวอย่างที่ 2.27
- 2.6.3 ฮิวริสติกและ Constraint Propagation — ตัวอย่างที่ 2.28

## ความรู้พื้นฐานที่ควรมี

- Python เบื้องต้น ได้แก่ ฟังก์ชัน ฟังก์ชันเวียนเกิด (Recursion) ลิสต์ และดิกชันนารี (ทบทวนได้ในภาคผนวก ก)
- อ่านหัวข้อ 2.5–2.6 ของหนังสือมาก่อน notebook นี้จะคำนวณตัวอย่างที่ 2.22–2.28 ซ้ำด้วยโค้ด

## วิธีใช้ notebook นี้

- กดปุ่ม **Open in Colab** ด้านบนเพื่อเปิดใน Google Colab แล้วรันทีละเซลล์จากบนลงล่างด้วย `Shift + Enter`
- ถ้ารันเซลล์ข้ามลำดับแล้วเกิดข้อผิดพลาด ให้เลือกเมนู **Runtime → Run all** เพื่อรันใหม่ทั้งหมด
- เซลล์ที่มี `# TODO` ในส่วน "แบบฝึกปฏิบัติ" คือส่วนที่ผู้เรียนต้องเติมโค้ดเอง
- ทั้ง notebook ใช้เวลารันรวมไม่ถึง 1 นาที

| ส่วน | หัวข้อ | ตัวอย่างในหนังสือ |
|:---:|---|---|
| 1 | Minimax บนต้นไม้เกมการเสนอราคาขายไฟฟ้า | 2.23 |
| 2 | Alpha-Beta Pruning บนต้นไม้เดียวกัน | 2.25 |
| 3 | AI เล่นเกม OX ด้วย Minimax และ Alpha-Beta | 2.22, 2.24 |
| 4 | CSP: การจัดตารางซ่อมบำรุงสถานีไฟฟ้าย่อย | 2.26, 2.27, 2.28 |
| 5 | CSP: การระบายสีแผนที่และซูโดกุ | หัวข้อ 2.6.1 |
| 6 | แบบฝึกปฏิบัติ: MRV และ Forward Checking | 2.28 |

## เตรียมเครื่องมือ

ปฏิบัติการนี้ใช้เฉพาะไลบรารีที่มีใน Google Colab อยู่แล้ว ได้แก่ `numpy` `pandas` (แสดงผลเป็นตาราง) `matplotlib` (วาดกราฟ) และ `networkx` (วาดกราฟข้อจำกัด) จึงไม่ต้องติดตั้งเพิ่ม

> **หมายเหตุ** ป้ายกำกับบนกราฟใช้ภาษาอังกฤษ เพราะ Colab ไม่มีฟอนต์ภาษาไทยติดตั้งไว้ ถ้าใช้ภาษาไทยบนกราฟจะแสดงเป็นสี่เหลี่ยม

In [ ]:
# ไม่ต้องติดตั้งไลบรารีเพิ่ม (ถ้ารันบนเครื่องตนเองแล้วขาดไลบรารี ให้รัน: !pip install -q numpy pandas matplotlib networkx)
import math
import random
import itertools
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

RANDOM_SEED = 42  # กำหนด seed ให้ผลการสุ่มเหมือนเดิมทุกครั้งที่รัน
plt.rcParams["figure.dpi"] = 100
INF = math.inf

print("pandas", pd.__version__, "| networkx", nx.__version__)

## ส่วนที่ 1 Minimax บนต้นไม้เกมการเสนอราคาขายไฟฟ้า (ตัวอย่างที่ 2.23)

บริษัทผลิตไฟฟ้า A (**MAX**) เลือกราคาเสนอขาย $a_1$ (ต่ำ) $a_2$ (กลาง) หรือ $a_3$ (สูง) จากนั้นบริษัทคู่แข่ง B (**MIN**) เห็นราคาแล้วเลือกตอบโต้ 3 แบบ ตัวเลขที่ใบไม้คือกำไรของ A (ล้านหน่วยเงิน ค่าสมมติ) และสมมติว่าเป็นเกมผลรวมศูนย์ (Zero-sum Game)

| การเดินของ A | ผลเมื่อ B ตอบโต้แบบที่ 1 | แบบที่ 2 | แบบที่ 3 |
|---|:---:|:---:|:---:|
| $a_1$ (ต่ำ) → โหนด B1 | 3 | 12 | 8 |
| $a_2$ (กลาง) → โหนด B2 | 2 | 4 | 6 |
| $a_3$ (สูง) → โหนด B3 | 14 | 5 | 2 |

เราแทนต้นไม้เกมด้วยดิกชันนารีที่บอกว่าแต่ละโหนดมีลูกอะไร ใบไม้เป็นตัวเลข (ค่าอรรถประโยชน์) แล้วเขียน Minimax แบบเวียนเกิดตามนิยามในหัวข้อ 2.5.2

In [ ]:
# ต้นไม้เกมของตัวอย่างที่ 2.23: โหนดภายในเป็นชื่อ ใบไม้เป็นตัวเลข
BID_TREE = {"A": ["B1", "B2", "B3"],
            "B1": [3, 12, 8],
            "B2": [2, 4, 6],
            "B3": [14, 5, 2]}
MOVE_NAMES = {"B1": "a1 (ต่ำ)", "B2": "a2 (กลาง)", "B3": "a3 (สูง)"}


def is_leaf(node):
    return isinstance(node, (int, float))


def minimax_tree(tree, node, is_max, values):
    """คืนค่ามินิแมกซ์ของ node และบันทึกค่าของทุกโหนดภายในลงใน values"""
    if is_leaf(node):
        return node
    child_values = [minimax_tree(tree, child, not is_max, values) for child in tree[node]]
    values[node] = max(child_values) if is_max else min(child_values)
    return values[node]


values = {}
root_value = minimax_tree(BID_TREE, "A", True, values)
for node in ["B1", "B2", "B3"]:
    print(f"{node} (MIN) = min{tuple(BID_TREE[node])} = {values[node]}")
best_child = max(BID_TREE["A"], key=lambda c: values[c])
print(f"ราก A (MAX) = max({values['B1']}, {values['B2']}, {values['B3']}) = {root_value}"
      f" → A ควรเลือก {MOVE_NAMES[best_child]}")

ผลตรงกับตัวอย่างที่ 2.23 คือ B1 = 3, B2 = 2, B3 = 2 และค่าที่ราก = 3 จึงเลือก $a_1$ แม้ $a_3$ มีโอกาสได้ 14 แต่ B ที่เล่นเก่งจะตอบโต้จนเหลือ 2

**คำถามชวนคิด** ถ้าใบไม้แรกของ B2 เปลี่ยนจาก 2 เป็น 5 การตัดสินใจของ A จะเปลี่ยนหรือไม่ (ลองแก้ `BID_TREE` แล้วรันใหม่)

## ส่วนที่ 2 Alpha-Beta Pruning บนต้นไม้เดียวกัน (ตัวอย่างที่ 2.25)

**การตัดแต่งกิ่งแอลฟา-บีตา (Alpha-Beta Pruning)** ส่งค่า $\alpha$ (ค่าที่ MAX รับประกันได้แล้ว) และ $\beta$ (ค่าที่ MIN รับประกันได้แล้ว) ลงไปตามต้นไม้ ที่โหนด MIN ถ้า $v \le \alpha$ ตัดลูกที่เหลือทิ้ง ที่โหนด MAX ถ้า $v \ge \beta$ ตัดลูกที่เหลือทิ้ง

ฟังก์ชันด้านล่างเขียนตามรหัสเทียม ALPHA-BETA-SEARCH ในหนังสือ และบันทึกแต่ละขั้นเป็นตารางแบบเดียวกับตัวอย่างที่ 2.25 พร้อมนับจำนวนใบไม้ที่ถูกประเมิน

In [ ]:
def fmt(x):
    """แสดง ±∞ ให้อ่านง่าย"""
    return {INF: "+∞", -INF: "−∞"}.get(x, x)


def alpha_beta_tree(tree, node, is_max, alpha, beta, log, counter):
    """Alpha-Beta บนต้นไม้แบบดิกชันนารี บันทึกแต่ละขั้นลงใน log และนับใบไม้ที่ประเมินใน counter"""
    if is_leaf(node):
        counter["leaves"] += 1
        return node
    v = -INF if is_max else INF
    children = tree[node]
    for i, child in enumerate(children):
        child_value = alpha_beta_tree(tree, child, not is_max, alpha, beta, log, counter)
        remaining = children[i + 1:]
        if is_max:
            if child_value > v:
                v = child_value
                alpha = max(alpha, v)
                action = f"รับค่า {child_value} จาก {child} ตั้ง α = {fmt(alpha)}"
            else:
                action = f"{child_value} จาก {child} ไม่มากกว่า {v} ค่าคงเดิม"
            prune = v >= beta
        else:
            if child_value < v:
                v = child_value
                beta = min(beta, v)
            prune = v <= alpha
            if prune and remaining:
                action = f"{v} ≤ α = {fmt(alpha)} → ตัดกิ่ง ไม่ดูใบไม้ {remaining}"
            elif prune:
                action = f"{v} ≤ α = {fmt(alpha)}? ใช่ (และไม่มีใบไม้เหลือแล้ว) คืนค่า {v}"
            elif not remaining:
                action = f"ครบ คืนค่า {v}"
            else:
                action = f"{v} ≤ α = {fmt(alpha)}? ไม่ → ทำต่อ"
        log.append({"โหนด": f"{node} ({'MAX' if is_max else 'MIN'})",
                    "ใบไม้ที่พิจารณา": child if is_leaf(child) else "—",
                    "v ของโหนด": v, "α": fmt(alpha), "β": fmt(beta), "การดำเนินการ": action})
        if prune:
            return v
    return v


def run_alpha_beta(tree):
    log, counter = [], {"leaves": 0}
    value = alpha_beta_tree(tree, "A", True, -INF, INF, log, counter)
    table = pd.DataFrame(log)
    table.index = range(1, len(table) + 1)
    table.index.name = "ขั้น"
    return value, counter["leaves"], table


value, leaves, table = run_alpha_beta(BID_TREE)
display(table)
total_leaves = sum(len(c) for c in BID_TREE.values() if is_leaf(c[0]))
print(f"ค่าที่ราก = {value} (เท่ากับ Minimax) ประเมินใบไม้ {leaves} จาก {total_leaves} ใบ")

ตารางตรงกับตัวอย่างที่ 2.25 คือที่ B2 ใบไม้แรกให้ $v = 2 \le \alpha = 3$ จึงตัดใบไม้ 4 และ 6 ทิ้ง ได้ค่าที่รากเท่า Minimax แต่ประเมินใบไม้เพียง 7 จาก 9 ใบ

### ผลของลำดับการเดิน

หนังสือบอกว่าถ้าลูกของ B3 เรียงเป็น 2, 5, 14 แทน จะตัดได้เพิ่มอีก 2 ใบ ลองตรวจด้วยโค้ด และลองเรียงทุกแบบที่เป็นไปได้ของลูก B2 และ B3 เพื่อดูว่าจำนวนใบไม้ที่ประเมินอยู่ในช่วงใด

In [ ]:
reordered = dict(BID_TREE, B3=[2, 5, 14])
value, leaves, _ = run_alpha_beta(reordered)
print(f"เรียงลูก B3 เป็น 2, 5, 14: ค่าที่ราก = {value} ประเมินใบไม้ {leaves} จาก 9 ใบ")

# ลองทุกลำดับของลูก B2 และ B3 (B1 คงเดิมเพื่อให้ α = 3 หลังโหนดแรก)
counts = []
for order_b2 in itertools.permutations(BID_TREE["B2"]):
    for order_b3 in itertools.permutations(BID_TREE["B3"]):
        _, leaves, _ = run_alpha_beta(dict(BID_TREE, B2=list(order_b2), B3=list(order_b3)))
        counts.append(leaves)
print("จำนวนใบไม้ที่ประเมินเมื่อเรียงลูก B2, B3 ทุกแบบ:", pd.Series(counts).value_counts().sort_index().to_dict())

**คำถามชวนคิด** ถ้าสลับให้ B2 ถูกพิจารณาก่อน B1 (รากมีลูกเรียงเป็น B2, B1, B3) จะประเมินใบไม้กี่ใบ ทำไมการพิจารณาการเดินที่ดีที่สุดก่อนจึงตัดกิ่งได้มากที่สุด

## ส่วนที่ 3 AI เล่นเกม OX ด้วย Minimax และ Alpha-Beta

### 3.1 นิยามเกมตามตารางที่ 2.13

แทนกระดาน 3 × 3 ด้วยสตริงยาว 9 ตัวอักษร (`.` คือช่องว่าง) ตำแหน่งเรียงจากซ้ายไปขวา บนลงล่าง ดังนี้

```text
 0 | 1 | 2
 3 | 4 | 5
 6 | 7 | 8
```

ฟังก์ชันแต่ละตัวตรงกับองค์ประกอบของเกมในตารางที่ 2.13 คือ TO-MOVE, ACTIONS, RESULT, IS-TERMINAL และ UTILITY (X ชนะ = +1, เสมอ = 0, O ชนะ = −1) โดย X เป็น MAX และเดินก่อน

In [ ]:
EMPTY_BOARD = "." * 9
LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8),      # แนวนอน
         (0, 3, 6), (1, 4, 7), (2, 5, 8),      # แนวตั้ง
         (0, 4, 8), (2, 4, 6)]                 # แนวทแยง


def to_move(board):
    """TO-MOVE: X เดินก่อน ถ้าจำนวน X เท่ากับ O แสดงว่าเป็นตาของ X"""
    return "X" if board.count("X") == board.count("O") else "O"


def actions(board):
    """ACTIONS: ช่องว่างทุกช่อง"""
    return [i for i, cell in enumerate(board) if cell == "."]


def result(board, move):
    """RESULT: กระดานใหม่หลังวางเครื่องหมายของผู้ที่ถึงตาที่ช่อง move"""
    return board[:move] + to_move(board) + board[move + 1:]


def winner(board):
    for a, b, c in LINES:
        if board[a] != "." and board[a] == board[b] == board[c]:
            return board[a]
    return None


def is_terminal(board):
    """IS-TERMINAL: มีผู้ชนะ หรือกระดานเต็ม"""
    return winner(board) is not None or "." not in board


def utility(board):
    """UTILITY มุมมองของ MAX (X): ชนะ +1, เสมอ 0, แพ้ -1"""
    return {"X": 1, "O": -1, None: 0}[winner(board)]


def show(board):
    rows = [" " + " | ".join(board[r * 3:r * 3 + 3]) for r in range(3)]
    print("\n---+---+---\n".join(rows))


demo = result(result(EMPTY_BOARD, 4), 0)      # X กลาง แล้ว O มุมซ้ายบน
show(demo)
print("ตาของ:", to_move(demo), "| การเดินที่ทำได้:", actions(demo))

### 3.2 ขนาดของต้นไม้เกม (ตัวอย่างที่ 2.22)

หนังสือให้ขอบบนของจำนวนโหนดที่ความลึก $d$ เป็น $9 \times 8 \times \cdots$ (ถ้ายังไม่มีใครชนะ) และบอกว่าจำนวนใบไม้จริงน้อยกว่า $9! = 362{,}880$ เพราะบางเกมจบก่อนกระดานเต็ม และสถานะกระดานที่ต่างกันจริงมีเพียง 5,478 สถานะ เราตรวจได้ด้วยการไล่สร้างต้นไม้เกมทั้งต้น

In [ ]:
def enumerate_game_tree():
    """ไล่ต้นไม้เกม OX ทั้งต้นแบบลึกก่อน นับโหนดแต่ละความลึก ใบไม้ และสถานะที่ต่างกัน"""
    nodes_per_depth = [0] * 10
    distinct_states = set()
    leaves = 0
    stack = [(EMPTY_BOARD, 0)]
    while stack:
        board, depth = stack.pop()
        nodes_per_depth[depth] += 1
        distinct_states.add(board)
        if is_terminal(board):
            leaves += 1
            continue
        for move in actions(board):
            stack.append((result(board, move), depth + 1))
    return nodes_per_depth, leaves, distinct_states


start = time.time()
nodes_per_depth, n_leaves, distinct_states = enumerate_game_tree()
rows = [{"ความลึก (ตา)": d, "ขอบบน 9!/(9-d)!": math.perm(9, d), "จำนวนโหนดจริง": n}
        for d, n in enumerate(nodes_per_depth)]
display(pd.DataFrame(rows).set_index("ความลึก (ตา)").T)
print(f"โหนดทั้งต้น = {sum(nodes_per_depth):,} | ใบไม้ (เกมที่จบ) = {n_leaves:,} (น้อยกว่า 9! = 362,880)")
print(f"สถานะกระดานที่ต่างกัน = {len(distinct_states):,} | ใช้เวลา {time.time() - start:.1f} วินาที")

ที่ความลึก 0–4 จำนวนโหนดเท่ากับขอบบนพอดี เพราะยังไม่มีใครเรียงครบ 3 ได้ (ผู้เดินก่อนต้องเดินอย่างน้อย 3 ครั้ง คือความลึก 5) ตั้งแต่ความลึก 5 เป็นต้นไปบางเกมจบแล้ว จำนวนจริงจึงน้อยกว่าขอบบน ใบไม้มี 255,168 ใบ และสถานะกระดานที่ต่างกันมี 5,478 สถานะตรงกับหนังสือ

### 3.3 Minimax และ Alpha-Beta สำหรับ OX พร้อมนับโหนด

ฟังก์ชันด้านล่างเขียนตามรหัสเทียม MINIMAX-SEARCH และ ALPHA-BETA-SEARCH โดยรวม MAX-VALUE กับ MIN-VALUE เป็นฟังก์ชันเดียวที่ตรวจว่าเป็นตาของใคร และนับจำนวนโหนดที่เยี่ยม (ทุกครั้งที่เรียกฟังก์ชัน) ไว้ใน `stats`

In [ ]:
def minimax_value(board, stats):
    """ค่ามินิแมกซ์ของกระดาน (มุมมองของ X)"""
    stats["nodes"] += 1
    if is_terminal(board):
        return utility(board)
    child_values = [minimax_value(result(board, m), stats) for m in actions(board)]
    return max(child_values) if to_move(board) == "X" else min(child_values)


def alpha_beta_value(board, alpha, beta, stats):
    """ค่ามินิแมกซ์ด้วย Alpha-Beta Pruning (ได้ค่าเดียวกับ minimax_value)"""
    stats["nodes"] += 1
    if is_terminal(board):
        return utility(board)
    if to_move(board) == "X":                      # MAX-VALUE
        v = -INF
        for m in actions(board):
            v = max(v, alpha_beta_value(result(board, m), alpha, beta, stats))
            alpha = max(alpha, v)
            if v >= beta:
                break                              # ตัดกิ่ง
        return v
    v = INF                                        # MIN-VALUE
    for m in actions(board):
        v = min(v, alpha_beta_value(result(board, m), alpha, beta, stats))
        beta = min(beta, v)
        if v <= alpha:
            break                                  # ตัดกิ่ง
    return v


def best_move(board, use_alpha_beta=True):
    """เลือกการเดินที่ดีที่สุดของผู้ที่ถึงตา คืน (การเดิน, ค่า, จำนวนโหนด)"""
    stats = {"nodes": 1}
    player = to_move(board)
    best, best_value = None, -INF if player == "X" else INF
    alpha, beta = -INF, INF
    for m in actions(board):
        child = result(board, m)
        if use_alpha_beta:
            v = alpha_beta_value(child, alpha, beta, stats)
        else:
            v = minimax_value(child, stats)
        if (player == "X" and v > best_value) or (player == "O" and v < best_value):
            best, best_value = m, v
        if use_alpha_beta:
            if player == "X":
                alpha = max(alpha, best_value)
            else:
                beta = min(beta, best_value)
    return best, best_value, stats["nodes"]


rows = []
test_boards = {"กระดานว่าง": EMPTY_BOARD,
               "X กลาง, O มุม (ตาของ X)": demo,
               "X ขู่ชนะ (ตาของ O)": "XX..O....",
               "X ชนะได้ทันที (ตาของ X)": "X.O.O.X.."}
for name, board in test_boards.items():
    for use_ab in [False, True]:
        start = time.time()
        move, value, nodes = best_move(board, use_ab)
        rows.append({"สถานการณ์": name, "วิธี": "Alpha-Beta" if use_ab else "Minimax",
                     "การเดินที่เลือก": move, "ค่า": value, "จำนวนโหนด": nodes,
                     "เวลา (วินาที)": round(time.time() - start, 3)})
node_table = pd.DataFrame(rows)
display(node_table)

In [ ]:
# สรุปอัตราส่วนจำนวนโหนดที่ประหยัดได้
pivot = node_table.pivot(index="สถานการณ์", columns="วิธี", values="จำนวนโหนด")
pivot["Alpha-Beta / Minimax (%)"] = (100 * pivot["Alpha-Beta"] / pivot["Minimax"]).round(1)
display(pivot)

**สิ่งที่ควรสังเกต**

- ทั้งสองวิธีได้ **ค่าเดียวกันทุกสถานการณ์** (กระดานว่างได้ค่า 0 แปลว่าถ้าทั้งสองฝ่ายเล่นดีที่สุด เกม OX จะเสมอเสมอ) แต่ Alpha-Beta เยี่ยมโหนดน้อยกว่ามาก จากกระดานว่าง Minimax เยี่ยมโหนดทั้งต้นคือ 549,946 โหนด ขณะที่ Alpha-Beta เยี่ยมเพียงไม่กี่เปอร์เซ็นต์ของจำนวนนั้น
- การเดินที่เลือกอาจต่างกันได้เมื่อมีหลายการเดินที่ค่าเท่ากัน เพราะ Alpha-Beta ไม่ได้คำนวณค่าที่แน่นอนของกิ่งที่ถูกตัด แต่ค่าที่รากเท่ากันเสมอ

### 3.4 ให้ AI เล่นกับผู้เล่นสุ่ม

ถ้า Minimax ถูกต้อง AI ต้อง **ไม่แพ้เลย** ไม่ว่าคู่แข่งจะเดินอย่างไร ลองให้ AI (Alpha-Beta) เล่นกับผู้เล่นที่สุ่มเดิน ทั้งกรณีที่ AI เป็น X และเป็น O

In [ ]:
def play_game(x_player, o_player):
    """เล่นเกมจนจบ player คือฟังก์ชันที่รับกระดานแล้วคืนการเดิน คืนผู้ชนะ ('X', 'O' หรือ 'draw')"""
    board = EMPTY_BOARD
    while not is_terminal(board):
        player = x_player if to_move(board) == "X" else o_player
        board = result(board, player(board))
    return winner(board) or "draw"


rng = random.Random(RANDOM_SEED)
random_player = lambda board: rng.choice(actions(board))
ai_player = lambda board: best_move(board, use_alpha_beta=True)[0]

start = time.time()
ai_as_x = pd.Series([play_game(ai_player, random_player) for _ in range(30)]).value_counts()
ai_as_o = pd.Series([play_game(random_player, ai_player) for _ in range(30)]).value_counts()
print("AI เป็น X พบผู้เล่นสุ่ม 30 เกม:", ai_as_x.to_dict())
print("AI เป็น O พบผู้เล่นสุ่ม 30 เกม:", ai_as_o.to_dict())
print("AI พบ AI:", play_game(ai_player, ai_player), f"| ใช้เวลา {time.time() - start:.1f} วินาที")

### 3.5 ฟังก์ชันประเมินและการค้นหาแบบจำกัดความลึก (ตัวอย่างที่ 2.24)

สำหรับเกมใหญ่ เราตัดการค้นหาที่ความลึกจำกัด แล้วใช้ **ฟังก์ชันประเมิน (Evaluation Function)** แทนค่าอรรถประโยชน์ ฟังก์ชันประเมินคลาสสิกของ OX คือ

$$\text{EVAL}(s) = (\text{จำนวนแถวที่ X ยังมีโอกาสเรียงครบ}) - (\text{จำนวนแถวที่ O ยังมีโอกาสเรียงครบ})$$

"ยังมีโอกาส" คือแถวนั้นยังไม่มีเครื่องหมายของฝ่ายตรงข้าม ตัวอย่างที่ 2.24 คำนวณกระดานที่ X อยู่กลางและ O อยู่มุมซ้ายบนได้ $5 - 4 = +1$

In [ ]:
def evaluate(board):
    """EVAL ของตัวอย่างที่ 2.24 (ถ้าเกมจบแล้ว ใช้ค่าอรรถประโยชน์จริงคูณ 100 เพื่อให้สำคัญกว่าทุกค่าประมาณ)"""
    if is_terminal(board):
        return 100 * utility(board)
    x_open = sum(all(board[i] != "O" for i in line) for line in LINES)
    o_open = sum(all(board[i] != "X" for i in line) for line in LINES)
    return x_open - o_open


def h_minimax(board, depth, limit, stats):
    """H-MINIMAX: ตัดที่ความลึก limit แล้วใช้ evaluate แทนค่าอรรถประโยชน์"""
    stats["nodes"] += 1
    if is_terminal(board) or depth == limit:
        return evaluate(board)
    child_values = [h_minimax(result(board, m), depth + 1, limit, stats) for m in actions(board)]
    return max(child_values) if to_move(board) == "X" else min(child_values)


def h_best_move(board, limit):
    stats = {"nodes": 1}
    scored = [(h_minimax(result(board, m), 1, limit, stats), m) for m in actions(board)]
    pick = max if to_move(board) == "X" else min
    value, move = pick(scored)
    return move, value, stats["nodes"]


show(demo)
x_open = sum(all(demo[i] != "O" for i in line) for line in LINES)
o_open = sum(all(demo[i] != "X" for i in line) for line in LINES)
print(f"แถวที่ X ยังมีโอกาส = {x_open}, แถวที่ O ยังมีโอกาส = {o_open}, EVAL = {evaluate(demo):+d}")

move, value, nodes = h_best_move(EMPTY_BOARD, limit=2)
print(f"\nจากกระดานว่าง H-MINIMAX ความลึก 2 ตา เลือกช่อง {move} (ค่าประเมิน {value:+d}) เยี่ยม {nodes} โหนด")
print("เทียบกับ Minimax เต็มต้นที่เยี่ยม 549,946 โหนด")

In [ ]:
# AI จำกัดความลึก 2 ตา เล่นกับผู้เล่นสุ่ม และกับ AI เต็มต้น
rng = random.Random(RANDOM_SEED)
shallow_ai = lambda board: h_best_move(board, limit=2)[0]
vs_random = pd.Series([play_game(shallow_ai, random_player) for _ in range(30)]).value_counts()
print("AI ความลึก 2 (X) พบผู้เล่นสุ่ม 30 เกม:", vs_random.to_dict())
print("AI ความลึก 2 (X) พบ AI เต็มต้น (O):", play_game(shallow_ai, ai_player))
print("AI เต็มต้น (X) พบ AI ความลึก 2 (O):", play_game(ai_player, shallow_ai))

### 3.6 ลองเล่นกับ AI ด้วยตนเอง

เซลล์ด้านล่างให้ผู้เรียนเล่นเป็น O กับ AI (X) โดยพิมพ์เลขช่อง 0–8 ลบเครื่องหมาย `#` หน้าบรรทัดสุดท้ายออกแล้วรัน (ปิดไว้เพื่อให้ notebook รันอัตโนมัติจนจบได้)

**คำถามชวนคิด**

1. AI เต็มต้นไม่เคยแพ้ แต่ทำไมจึงไม่ชนะผู้เล่นสุ่มทุกเกม
2. EVAL ในตัวอย่างที่ 2.24 ไม่สนใจว่าแถวหนึ่งมีเครื่องหมายแล้วกี่ตัว ลองเสนอฟังก์ชันประเมินที่ดีกว่านี้

In [ ]:
def play_human_vs_ai():
    """เล่นเป็น O กับ AI (X) พิมพ์เลขช่อง 0-8"""
    board = EMPTY_BOARD
    while not is_terminal(board):
        if to_move(board) == "X":
            board = result(board, best_move(board)[0])
        else:
            show(board)
            move = int(input(f"ช่องว่างที่เลือกได้ {actions(board)}: "))
            if move in actions(board):
                board = result(board, move)
    show(board)
    print("ผล:", winner(board) or "เสมอ")


# play_human_vs_ai()

## ส่วนที่ 4 CSP: การจัดตารางซ่อมบำรุงสถานีไฟฟ้าย่อย (ตัวอย่างที่ 2.26–2.28)

### 4.1 นิยามปัญหา

สถานีไฟฟ้าย่อย (Substation) มีหม้อแปลงกำลังสองตัว (TR1, TR2) ที่จ่ายโหลดร่วมกัน ต้องซ่อมบำรุงหม้อแปลงทั้งสอง ซ่อมเบรกเกอร์ และทดสอบรีเลย์ป้องกันให้เสร็จภายใน 3 วัน (1 = จันทร์, 2 = อังคาร, 3 = พุธ)

| องค์ประกอบ | นิยาม |
|---|---|
| ตัวแปร | $T_1$, $T_2$ (วันซ่อม TR1, TR2), $B$ (วันซ่อมเบรกเกอร์), $R$ (วันทดสอบรีเลย์) |
| โดเมน | ทุกตัวแปร $\in \{1, 2, 3\}$ |
| C1 | $T_1 \ne T_2$ — ห้ามปลดหม้อแปลงทั้งสองตัวพร้อมกัน |
| C2 | $B < R$ — ซ่อมเบรกเกอร์ก่อนวันทดสอบรีเลย์ |
| C3, C4 | $B \ne T_1$, $B \ne T_2$ — ทีมช่างกำลังมีทีมเดียว |
| C5 | $R \ne T_1$ — ชุดเครื่องมือทดสอบมีชุดเดียว |

เราเขียนคลาส `CSP` ทั่วไปที่เก็บตัวแปร โดเมน ข้อจำกัดทวิภาค (Binary Constraint) และชื่อข้อจำกัด แล้วใช้คลาสเดียวกันกับทุกปัญหาในส่วนที่ 4–6

In [ ]:
class CSP:
    """CSP ที่มีเฉพาะข้อจำกัดทวิภาค

    constraints: ลิสต์ของ (ชื่อ, X, Y, ฟังก์ชัน f(x, y) -> bool) แทนข้อจำกัดระหว่างตัวแปร X กับ Y
    """

    def __init__(self, variables, domains, constraints):
        self.variables = list(variables)
        self.domains = {v: list(domains[v]) for v in self.variables}
        self.neighbors = {v: [] for v in self.variables}
        self.checks = {}                                   # (X, Y) -> [(ชื่อ, f)] ใช้ f(ค่าของ X, ค่าของ Y)
        for name, x, y, f in constraints:
            if y not in self.neighbors[x]:
                self.neighbors[x].append(y)
                self.neighbors[y].append(x)
            self.checks.setdefault((x, y), []).append((name, f))
            self.checks.setdefault((y, x), []).append((name, lambda b, a, f=f: f(a, b)))

    def satisfied(self, x, x_value, y, y_value):
        """ข้อจำกัดทุกข้อระหว่าง x กับ y เป็นจริงหรือไม่"""
        return all(f(x_value, y_value) for _, f in self.checks.get((x, y), []))

    def conflicts(self, var, value, assignment):
        """ชื่อข้อจำกัดที่ถูกละเมิด ถ้ากำหนด var = value ร่วมกับ assignment"""
        names = []
        for other in self.neighbors[var]:
            if other in assignment:
                names += [n for n, f in self.checks[(var, other)] if not f(value, assignment[other])]
        return names

    def is_solution(self, assignment):
        return len(assignment) == len(self.variables) and all(
            not self.conflicts(v, assignment[v], assignment) for v in self.variables)


def make_maintenance_csp():
    not_equal = lambda a, b: a != b
    constraints = [("C1", "T1", "T2", not_equal),
                   ("C2", "B", "R", lambda b, r: b < r),
                   ("C3", "B", "T1", not_equal),
                   ("C4", "B", "T2", not_equal),
                   ("C5", "R", "T1", not_equal)]
    variables = ["T1", "T2", "B", "R"]
    return CSP(variables, {v: [1, 2, 3] for v in variables}, constraints)


maintenance = make_maintenance_csp()
print("กราฟข้อจำกัด (ข้างเคียงของแต่ละตัวแปร):", maintenance.neighbors)
for example in [{"T1": 1, "T2": 2, "B": 3, "R": 3}, {"T1": 2, "T2": 3, "B": 1, "R": 3}]:
    broken = sorted({n for v in example for n in maintenance.conflicts(v, example[v], example)})
    print(example, "→", "เป็นคำตอบ" if not broken else f"ละเมิด {broken}")

In [ ]:
# วาดกราฟข้อจำกัด (รูปที่ 2.12)
def draw_constraint_graph(csp, positions, edge_labels, colors=None, title=""):
    graph = nx.Graph()
    graph.add_nodes_from(csp.variables)
    graph.add_edges_from((x, y) for x in csp.variables for y in csp.neighbors[x] if x < y)
    fig, ax = plt.subplots(figsize=(5, 4))
    node_colors = [colors[v] for v in graph.nodes] if colors else "#1F4E79"
    nx.draw_networkx(graph, positions, ax=ax, node_color=node_colors, node_size=1300,
                     font_color="white", font_weight="bold")
    nx.draw_networkx_edge_labels(graph, positions, edge_labels=edge_labels, ax=ax, font_size=8)
    ax.set_title(title)
    ax.margins(0.15)
    ax.axis("off")
    plt.show()


MAINT_POS = {"T1": (-1, 0), "B": (0, 1), "T2": (1, 0), "R": (0, -1)}
MAINT_EDGE_LABELS = {("T1", "T2"): "C1: T1 != T2", ("B", "R"): "C2: B < R", ("B", "T1"): "C3: !=",
                     ("B", "T2"): "C4: !=", ("R", "T1"): "C5: !="}
draw_constraint_graph(maintenance, MAINT_POS, MAINT_EDGE_LABELS, title="Maintenance scheduling CSP")

### 4.2 นับคำตอบทั้งหมด

ปัญหามีการกำหนดค่าสมบูรณ์ $3^4 = 81$ แบบ เล็กพอที่จะไล่ทุกแบบได้ หนังสือนับด้วยมือได้ **3 คำตอบ** ตรวจด้วยโค้ด

In [ ]:
DAY_NAMES = {1: "จันทร์", 2: "อังคาร", 3: "พุธ"}

all_assignments = [dict(zip(maintenance.variables, values))
                   for values in itertools.product([1, 2, 3], repeat=len(maintenance.variables))]
solutions = [a for a in all_assignments if maintenance.is_solution(a)]
print(f"การกำหนดค่าสมบูรณ์ {len(all_assignments)} แบบ เป็นคำตอบ {len(solutions)} แบบ")
display(pd.DataFrame([{v: f"{a[v]} ({DAY_NAMES[a[v]]})" for v in maintenance.variables} for a in solutions],
                     index=[f"คำตอบที่ {i}" for i in range(1, len(solutions) + 1)]))

ได้ 3 คำตอบตรงกับตัวอย่างที่ 2.26 คือ $B = 1$ สองคำตอบ ($T_1, T_2, R$ = 2, 3, 3 และ 3, 2, 2) และ $B = 2$ หนึ่งคำตอบ ($T_1, T_2, R$ = 1, 3, 3)

### 4.3 Backtracking Search พื้นฐาน (ตัวอย่างที่ 2.27)

ฟังก์ชัน `backtracking_search` เขียนตามรหัสเทียม BACKTRACK ในหัวข้อ 2.6.2 โดยแยก 3 ช่องที่ปรับปรุงได้ออกเป็นพารามิเตอร์

- `select_variable` — SELECT-UNASSIGNED-VARIABLE (ค่าเริ่มต้น: ตัวแรกตามลำดับที่กำหนด)
- `order_values` — ORDER-DOMAIN-VALUES (ค่าเริ่มต้น: ตามลำดับในโดเมน ซึ่งสำหรับตัวเลขคือจากน้อยไปมาก)
- `inference` — INFERENCE (ค่าเริ่มต้น: ไม่ทำอะไร) ต้องคืนรายการ `(ตัวแปร, ค่า)` ที่ตัดออกจากโดเมน หรือ `None` ถ้าพบโดเมนว่าง

ฟังก์ชันนับสองค่าเพื่อวัดประสิทธิภาพ

- **จำนวนการลองค่า (tries)** — นับทุกครั้งที่นำค่าหนึ่งมาตรวจกับข้อจำกัด (ตรงกับ "การลองค่า 15 ครั้ง" ในตัวอย่างที่ 2.27)
- **จำนวนการย้อนรอย (backtracks)** — นับเมื่อพบ **ทางตัน** คือตัวแปรที่ไม่มีค่าใดผ่านทั้งการตรวจข้อจำกัดและ INFERENCE เลย จึงต้องย้อนกลับ การย้อนต่อเนื่องหลายระดับจากทางตันเดียวนับเป็น 1 ครั้ง ตรงกับวิธีนับในหนังสือ ("ย้อนรอย 1 ครั้ง จาก R ผ่าน B กลับไปถึง $T_2$")

In [ ]:
def select_first_unassigned(csp, assignment, domains):
    """เลือกตัวแปรตัวแรกที่ยังไม่มีค่าตามลำดับใน csp.variables"""
    return next(v for v in csp.variables if v not in assignment)


def order_domain(csp, var, assignment, domains):
    """ลองค่าตามลำดับในโดเมนเดิม (ตัวเลขเรียงจากน้อยไปมาก)"""
    return sorted(domains[var], key=csp.domains[var].index)


def no_inference(csp, var, value, assignment, domains):
    """ไม่อนุมานอะไรเพิ่ม (ไม่ตัดค่าใดออก)"""
    return []


class TooManyTries(Exception):
    """ใช้หยุดการค้นหาเมื่อลองค่าเกินจำนวนที่กำหนด"""


def backtracking_search(csp, select_variable=select_first_unassigned, order_values=order_domain,
                        inference=no_inference, domains=None, trace=False, max_tries=None):
    """คืน (คำตอบหรือ None, สถิติ, ตารางบันทึก)

    max_tries: ถ้ากำหนด จะหยุดเมื่อลองค่าเกินจำนวนนี้ และตั้ง stats["stopped"] = True
    """
    domains = {v: list(d) for v, d in (domains or csp.domains).items()}   # โดเมนปัจจุบัน (ถูกตัดได้)
    stats = {"tries": 0, "backtracks": 0}
    log = []

    def record(var, value, outcome, assignment):
        if trace:
            log.append({"ขั้น": stats["tries"] if value is not None else "—",
                        "การกำหนดค่าที่ลอง": f"{var} = {value}" if value is not None else "—",
                        "ผล": outcome,
                        "การกำหนดค่าปัจจุบัน": ", ".join(f"{k}={v}" for k, v in assignment.items())})

    def backtrack(assignment):
        if len(assignment) == len(csp.variables):
            return dict(assignment)
        var = select_variable(csp, assignment, domains)
        went_deeper = False
        for value in order_values(csp, var, assignment, domains):
            stats["tries"] += 1
            if max_tries is not None and stats["tries"] > max_tries:
                raise TooManyTries
            broken = csp.conflicts(var, value, assignment)
            if broken:
                record(var, value, f"✗ ขัด {', '.join(broken)}", assignment)
                continue
            assignment[var] = value
            removed = inference(csp, var, value, assignment, domains)
            if removed is None:                                   # INFERENCE พบโดเมนว่าง
                del assignment[var]
                record(var, value, "✗ Forward Checking พบโดเมนว่าง", assignment)
                continue
            record(var, value, "✓", assignment)
            went_deeper = True
            solution = backtrack(assignment)
            if solution is not None:
                return solution
            for other, other_value in removed:                    # คืนค่าที่ INFERENCE ตัดออก
                domains[other].append(other_value)
            del assignment[var]
        if not went_deeper:
            stats["backtracks"] += 1                              # ทางตัน
        record(var, None, f"{var} ไม่มีค่าเหลือ → ย้อนรอย", assignment)
        return None

    try:
        solution = backtrack({})
    except TooManyTries:
        solution, stats["tries"], stats["stopped"] = None, max_tries, True
    return solution, stats, pd.DataFrame(log)


solution, stats, log = backtracking_search(maintenance, trace=True)
display(log.set_index("ขั้น"))
print("คำตอบ:", solution, "| สถิติ:", stats)

ตารางตรงกับตัวอย่างที่ 2.27 คือได้คำตอบ $\{T_1 = 1, T_2 = 3, B = 2, R = 3\}$ ด้วยการลองค่า 15 ครั้ง และย้อนรอย 1 ครั้ง (ทางตันที่ $R$ ทำให้ย้อนผ่าน $B$ กลับไปถึง $T_2$)

> **หมายเหตุ** คอลัมน์ "ขั้น" ในตารางนี้นับเฉพาะการลองค่า แถวย้อนรอยจึงไม่มีหมายเลข (แสดงเป็น —) ส่วนหนังสือนับแถว "B ไม่มีค่าเหลือ" เป็นขั้นที่ 10 ดังนั้นขั้นที่ 10–15 ในตารางนี้ตรงกับขั้นที่ 11–16 ในหนังสือ และในแถวที่ลอง $R = 1$ ตารางนี้แสดงข้อจำกัดที่ขัดครบทุกข้อ (C2 และ C5) ขณะที่หนังสือแสดงเฉพาะ C2

ขั้นที่ 6–9 และการย้อนรอยที่ตามมาเป็นงานที่สูญเปล่า เพราะเมื่อกำหนด $B = 3$ แล้ว $R$ ต้องมากกว่า 3 ซึ่งเป็นไปไม่ได้

### 4.4 AC-3 และ LCV (ตัวอย่างที่ 2.28 ขั้นที่ 1 และ 3)

**AC-3** ทำให้ทุกเส้นเชื่อมสอดคล้อง (Arc Consistency) ก่อนเริ่มค้นหา เซลล์ถัดไปเขียนตามรหัสเทียม AC-3 และ REVISE ในหนังสือ แล้วแสดงเฉพาะเส้นเชื่อมที่ทำให้โดเมนเปลี่ยน

In [ ]:
def revise(csp, domains, xi, xj):
    """ตัดค่าใน Di ที่ไม่มีค่าใดใน Dj ทำให้ข้อจำกัดเป็นจริง คืนรายการค่าที่ตัดออก"""
    removed = [x for x in domains[xi] if not any(csp.satisfied(xi, x, xj, y) for y in domains[xj])]
    for x in removed:
        domains[xi].remove(x)
    return removed


def ac3(csp, domains=None):
    """คืน (โดเมนใหม่หรือ None ถ้าพบโดเมนว่าง, ตารางบันทึก)"""
    domains = {v: list(d) for v, d in (domains or csp.domains).items()}
    queue = [(xi, xj) for xi in csp.variables for xj in csp.neighbors[xi]]
    log = []
    while queue:
        xi, xj = queue.pop(0)
        removed = revise(csp, domains, xi, xj)
        if removed:
            if not domains[xi]:
                return None, pd.DataFrame(log)
            added = [(xk, xi) for xk in csp.neighbors[xi] if xk != xj]
            queue += added
            log.append({"เส้นเชื่อมที่ตรวจ": f"({xi}, {xj})", "ค่าที่ตัด": f"{xi} = {removed}",
                        "โดเมนหลังตรวจ": f"D_{xi} = {domains[xi]}",
                        "เพิ่มลงคิว": ", ".join(f"({a}, {b})" for a, b in added)})
    return domains, pd.DataFrame(log)


ac3_domains, ac3_log = ac3(maintenance)
display(ac3_log)
print("โดเมนหลัง AC-3:", ac3_domains)

ได้ $D_B = \{1, 2\}$ และ $D_R = \{2, 3\}$ ตรงกับขั้นที่ 1 ของตัวอย่างที่ 2.28 ค่า $B = 3$ ที่เป็นต้นเหตุของการย้อนรอยถูกตัดออกตั้งแต่ก่อนเริ่มค้นหา

**ค่าที่จำกัดน้อยที่สุด (Least Constraining Value: LCV)** เรียงค่าโดยลองค่าที่ตัดค่าออกจากโดเมนของตัวแปรข้างเคียงที่ยังไม่มีค่าน้อยที่สุดก่อน (ถ้าเสมอกันเลือกค่าน้อยกว่า) ฟังก์ชันนี้ใช้เป็น `order_values` ได้ และจะใช้ในแบบฝึกปฏิบัติ

In [ ]:
def count_eliminations(csp, var, value, assignment, domains):
    """จำนวนค่าที่จะถูกตัดออกจากโดเมนของข้างเคียงที่ยังไม่มีค่า ถ้ากำหนด var = value"""
    return sum(1 for other in csp.neighbors[var] if other not in assignment
               for other_value in domains[other] if not csp.satisfied(var, value, other, other_value))


def order_lcv(csp, var, assignment, domains):
    """LCV: เรียงค่าตามจำนวนค่าที่ตัดจากข้างเคียง (น้อยก่อน) ถ้าเสมอเรียงตามค่า"""
    return sorted(domains[var], key=lambda value: (count_eliminations(csp, var, value, assignment, domains),
                                                   csp.domains[var].index(value)))


for value in ac3_domains["B"]:
    print(f"B = {value} ตัดค่าจากข้างเคียง {count_eliminations(maintenance, 'B', value, {}, ac3_domains)} ค่า")
print("LCV เรียงค่าของ B เป็น:", order_lcv(maintenance, "B", {}, ac3_domains))

ผลตรงกับขั้นที่ 3 ของตัวอย่างที่ 2.28 ($B = 1$ ตัด 2 ค่า, $B = 2$ ตัด 3 ค่า จึงลอง $B = 1$ ก่อน) ส่วนที่เหลือของตัวอย่างที่ 2.28 ต้องใช้ MRV และ Forward Checking ซึ่งผู้เรียนจะเขียนเองในแบบฝึกปฏิบัติ

**คำถามชวนคิด** ถ้าเพิ่มข้อจำกัด "ห้ามทดสอบรีเลย์วันเดียวกับซ่อม TR2" ($R \ne T_2$) จะเหลือคำตอบกี่คำตอบ (ลองเพิ่มใน `make_maintenance_csp` แล้วรันส่วน 4.2 ใหม่)

## ส่วนที่ 5 CSP: การระบายสีแผนที่และซูโดกุ

### 5.1 การระบายสีแผนที่ออสเตรเลีย

ปัญหาการระบายสีแผนที่ (Map Coloring) ของ Russell และ Norvig (2021) ให้ระบายสีรัฐและดินแดนของออสเตรเลีย 7 แห่งด้วยสีแดง เขียว น้ำเงิน โดยพื้นที่ที่มีพรมแดนติดกันต้องสีต่างกัน (WA = Western Australia, NT = Northern Territory, SA = South Australia, Q = Queensland, NSW = New South Wales, V = Victoria, T = Tasmania ซึ่งเป็นเกาะ ไม่ติดกับรัฐใด)

ปัญหาเดียวกันนี้พบในงานไฟฟ้าและสื่อสาร เช่น การกำหนดช่องความถี่วิทยุให้สถานีที่อยู่ใกล้กันไม่รบกวนกัน ซึ่งสถานีคือพื้นที่ และช่องความถี่คือสี

In [ ]:
AUSTRALIA_BORDERS = [("WA", "NT"), ("WA", "SA"), ("NT", "SA"), ("NT", "Q"), ("SA", "Q"),
                     ("SA", "NSW"), ("SA", "V"), ("Q", "NSW"), ("NSW", "V")]
COLORS = ["red", "green", "blue"]


def make_australia_csp():
    regions = ["WA", "NT", "Q", "NSW", "V", "SA", "T"]
    constraints = [(f"{a}≠{b}", a, b, lambda x, y: x != y) for a, b in AUSTRALIA_BORDERS]
    return CSP(regions, {r: COLORS for r in regions}, constraints)


australia = make_australia_csp()
solution, stats, log = backtracking_search(australia, trace=True)
print("คำตอบ:", solution, "| สถิติ:", stats)
display(log.set_index("ขั้น"))

AUS_POS = {"WA": (0, 1), "NT": (1, 2), "SA": (1.3, 0.8), "Q": (2.3, 1.8), "NSW": (2.4, 0.6),
           "V": (2, -0.2), "T": (2.2, -1)}
draw_constraint_graph(australia, AUS_POS, {}, colors={r: {"red": "#C0392B", "green": "#3A7D44",
                      "blue": "#1F4E79"}[solution[r]] for r in solution}, title="Australia map coloring")

สังเกตว่า SA ซึ่งติดกับพื้นที่อื่นถึง 5 แห่ง ถูกกำหนดค่าเกือบสุดท้ายตามลำดับตัวแปรที่ตั้งไว้ ถ้ามันเหลือค่าให้เลือกไม่ได้ ก็ต้องย้อนรอย ฮิวริสติก MRV และ Degree ในแบบฝึกปฏิบัติจะเลือก SA ก่อน

### 5.2 ซูโดกุ

**ซูโดกุ (Sudoku)** มีตัวแปร 81 ช่อง โดเมน {1, …, 9} และข้อจำกัด Alldiff 27 ข้อ (แถว 9 คอลัมน์ 9 กล่อง 3 × 3 อีก 9) ในที่นี้แปลง Alldiff แต่ละข้อเป็นข้อจำกัด "≠" ระหว่างทุกคู่ช่องในกลุ่มเดียวกัน (แต่ละช่องจึงมีข้างเคียง 20 ช่อง) ส่วนช่องที่โจทย์ให้ตัวเลขมาแล้วเป็นข้อจำกัดเอกภาค (Unary Constraint) จึงตัดโดเมนให้เหลือค่าเดียวตั้งแต่ต้น

`EASY_PUZZLE` เป็นตัวอย่างซูโดกุที่เผยแพร่ในวิกิพีเดีย (บทความ "Sudoku") ซึ่งคนทั่วไปแก้ได้โดยไม่ต้องเดา ส่วน `HARD_PUZZLE` เป็นโจทย์ยากจากบทความ "Solving Every Sudoku Puzzle" ของ Peter Norvig ใช้ในแบบฝึกปฏิบัติ

ลองแก้ `EASY_PUZZLE` ด้วย Backtracking พื้นฐาน โดยจำกัดการลองค่าไว้ไม่เกิน 300,000 ครั้ง (ใช้เวลาประมาณ 2–5 วินาที)

In [ ]:
EASY_PUZZLE = ("53..7...."
               "6..195..."
               ".98....6."
               "8...6...3"
               "4..8.3..1"
               "7...2...6"
               ".6....28."
               "...419..5"
               "....8..79")
HARD_PUZZLE = ("4.....8.5"
               ".3......."
               "...7....."
               ".2.....6."
               "....8.4.."
               "....1...."
               "...6.3.7."
               "5..2....."
               "1.4......")


def make_sudoku_csp(puzzle):
    cells_ = [(r, c) for r in range(9) for c in range(9)]
    domains = {}
    for (r, c), ch in zip(cells_, puzzle):
        domains[(r, c)] = [int(ch)] if ch != "." else list(range(1, 10))   # ข้อจำกัดเอกภาค
    constraints = []
    for (r1, c1), (r2, c2) in itertools.combinations(cells_, 2):
        same_box = (r1 // 3, c1 // 3) == (r2 // 3, c2 // 3)
        if r1 == r2 or c1 == c2 or same_box:
            constraints.append(("≠", (r1, c1), (r2, c2), lambda x, y: x != y))
    return CSP(cells_, domains, constraints)


def show_sudoku(assignment_or_puzzle):
    """พิมพ์ตารางซูโดกุจากสตริงโจทย์หรือดิกชันนารีคำตอบ"""
    if isinstance(assignment_or_puzzle, str):
        get = lambda r, c: assignment_or_puzzle[r * 9 + c]
    else:
        get = lambda r, c: str(assignment_or_puzzle[(r, c)])
    for r in range(9):
        if r in (3, 6):
            print("------+-------+------")
        print(" ".join(get(r, c) + (" |" if c in (2, 5) else "") for c in range(9)))


easy_sudoku = make_sudoku_csp(EASY_PUZZLE)
print("ช่องว่าง:", EASY_PUZZLE.count("."), "ช่อง | ข้างเคียงของช่อง (0, 0):",
      len(easy_sudoku.neighbors[(0, 0)]), "ช่อง\n")
show_sudoku(EASY_PUZZLE)

start = time.time()
solution, stats, _ = backtracking_search(easy_sudoku, max_tries=300_000)
print(f"\nBacktracking พื้นฐาน: สถิติ {stats} ใช้เวลา {time.time() - start:.1f} วินาที")
print("พบคำตอบ" if solution else "ลองค่าครบ 300,000 ครั้งแล้วยังไม่พบคำตอบ")

Backtracking พื้นฐานลองค่าครบ 300,000 ครั้งแล้วยังไม่พบคำตอบ (เมื่อทดลองโดยไม่จำกัด ต้องลองค่าประมาณ 66 ล้านครั้งและใช้เวลาหลายนาที) ทั้งที่คนทั่วไปแก้โจทย์นี้ได้โดยแทบไม่ต้องเดา สาเหตุคือ Backtracking พื้นฐานกำหนดค่าช่องตามลำดับแถว และพบความขัดแย้งก็ต่อเมื่อเดินไปถึงช่องที่ขัดเท่านั้น เพราะคนจะเลือกช่องที่เหลือตัวเลือกน้อยก่อน (MRV) และตัดตัวเลือกของช่องข้างเคียงทันทีที่เขียนตัวเลข (Forward Checking) ซึ่งเป็นสิ่งที่ผู้เรียนจะเพิ่มในแบบฝึกปฏิบัติ

**คำถามชวนคิด** `EASY_PUZZLE` มีช่องว่าง 51 ช่อง ถ้าไล่ทุกการกำหนดค่าจะมีกี่แบบ เทียบกับ 66 ล้านครั้งที่ Backtracking พื้นฐานใช้ แสดงว่าการตรวจข้อจำกัดทันทีที่กำหนดค่าช่วยได้มากเพียงใด

## ส่วนที่ 6 แบบฝึกปฏิบัติ: MRV และ Forward Checking

### ข้อ 1 ฮิวริสติก MRV (พร้อม Degree Heuristic)

เขียน `select_mrv(csp, assignment, domains)` ให้คืนตัวแปรที่ยังไม่มีค่าซึ่ง

1. มีค่าในโดเมนปัจจุบัน `domains[v]` เหลือ **น้อยที่สุด** (MRV)
2. ถ้าเสมอกัน เลือกตัวที่มี **ข้างเคียงที่ยังไม่มีค่ามากที่สุด** (Degree Heuristic)
3. ถ้ายังเสมอ เลือกตัวที่มาก่อนตามลำดับใน `csp.variables`

ใบ้: ใช้ `min(..., key=lambda v: (ค่าที่ 1, ค่าที่ 2))` โดยให้ค่าที่ 2 เป็นค่าติดลบของจำนวนข้างเคียงที่ยังไม่มีค่า (`min` คืนตัวแรกเมื่อ key เท่ากัน จึงได้ข้อ 3 อัตโนมัติ)

In [ ]:
def select_mrv(csp, assignment, domains):
    """MRV + Degree Heuristic"""
    unassigned = [v for v in csp.variables if v not in assignment]
    # TODO: คืนตัวแปรใน unassigned ที่ len(domains[v]) น้อยที่สุด
    #       ถ้าเสมอ เลือกตัวที่มีข้างเคียงที่ยังไม่มีค่ามากที่สุด
    return unassigned[0]

### ข้อ 2 Forward Checking

เขียน `forward_checking(csp, var, value, assignment, domains)` ซึ่งถูกเรียกหลังกำหนด `var = value` แล้ว ให้

1. สำหรับข้างเคียง `other` ของ `var` ที่ **ยังไม่มีค่า** ตัดค่า `other_value` ทุกค่าใน `domains[other]` ที่ `csp.satisfied(var, value, other, other_value)` เป็นเท็จ ออกจากโดเมน และจดคู่ `(other, other_value)` ไว้ในลิสต์ `removed`
2. ถ้าโดเมนของข้างเคียงตัวใดว่าง ให้ **คืนค่าที่ตัดไปทั้งหมดกลับเข้าโดเมน** แล้วคืน `None` (บอกว่าเป็นทางตัน)
3. มิฉะนั้นคืน `removed` (ฟังก์ชัน `backtracking_search` จะใช้ลิสต์นี้คืนค่าเมื่อย้อนรอย)

In [ ]:
def forward_checking(csp, var, value, assignment, domains):
    """ตัดค่าที่ขัดกับ var = value ออกจากโดเมนของข้างเคียงที่ยังไม่มีค่า"""
    removed = []
    # TODO: วนข้างเคียงที่ยังไม่มีค่า ตัดค่าที่ขัดข้อจำกัดออกจาก domains[other] และจดลง removed
    # TODO: ถ้า domains[other] ว่าง ให้คืนค่าทั้งหมดใน removed กลับเข้าโดเมน แล้ว return None
    return removed

In [ ]:
# เซลล์ตรวจคำตอบข้อ 1-2 (ไม่ต้องแก้): ทำซ้ำตัวอย่างที่ 2.28 ด้วย AC-3 + MRV/Degree + LCV + Forward Checking
probe = {v: list(d) for v, d in ac3_domains.items()}
mrv_pick = select_mrv(maintenance, {}, probe)
fc_probe = forward_checking(maintenance, "B", 3, {"B": 3}, probe)

if mrv_pick != "B":
    print(f"ข้อ 1: ยังไม่ผ่าน — หลัง AC-3 ตัวแปรแรกควรเป็น B (MRV เสมอกับ R แล้วตัดสินด้วย Degree) แต่ได้ {mrv_pick}")
elif fc_probe is not None or probe != ac3_domains:
    print("ข้อ 2: ยังไม่ผ่าน — กำหนด B = 3 แล้วโดเมนของ R ต้องว่าง ฟังก์ชันต้องคืน None และคืนค่าโดเมนให้เหมือนเดิม")
else:
    solution, stats, log = backtracking_search(maintenance, select_variable=select_mrv, order_values=order_lcv,
                                               inference=forward_checking, domains=ac3_domains, trace=True)
    display(log.set_index("ขั้น"))
    expected = {"B": 1, "T1": 2, "R": 3, "T2": 3}
    ok = solution == expected and stats == {"tries": 4, "backtracks": 0}
    print("คำตอบ:", solution, "| สถิติ:", stats)
    print("ข้อ 1-2:", "ผ่าน — ตรงกับตัวอย่างที่ 2.28 (กำหนดค่า 4 ครั้ง ไม่ย้อนรอย)" if ok else "ยังไม่ผ่าน")

### ข้อ 3 เปรียบเทียบจำนวนการลองค่าและการย้อนรอย

เติมโค้ดให้รัน `backtracking_search` กับทุกปัญหาใน `PROBLEMS` ด้วยทุกชุดเทคนิคใน `METHODS` แล้วเก็บผลลงลิสต์ `comparison_rows` หนึ่งแถวต่อหนึ่งคู่ (ปัญหา, ชุดเทคนิค) ประกอบด้วย `"ปัญหา"`, `"เทคนิค"`, `"tries"`, `"backtracks"`, `"ถูกต้อง"` (ใช้ `csp.is_solution`) และ `"เวลา (วินาที)"`

ส่ง `max_tries=MAX_TRIES` ทุกครั้ง เพื่อหยุดชุดเทคนิคที่ช้าเกินไป และเก็บคอลัมน์ `"เสร็จ"` (`not stats.get("stopped", False)`) ไว้ด้วย เซลล์นี้ใช้เวลาประมาณ 10 วินาที

In [ ]:
PROBLEMS = {"ซ่อมบำรุง": make_maintenance_csp(), "ออสเตรเลีย": make_australia_csp(),
            "ซูโดกุ (ง่าย)": make_sudoku_csp(EASY_PUZZLE), "ซูโดกุ (ยาก)": make_sudoku_csp(HARD_PUZZLE)}
METHODS = {
    "พื้นฐาน": dict(),
    "MRV": dict(select_variable=select_mrv),
    "FC": dict(inference=forward_checking),
    "MRV + FC": dict(select_variable=select_mrv, inference=forward_checking),
    "MRV + FC + LCV": dict(select_variable=select_mrv, inference=forward_checking, order_values=order_lcv),
}
MAX_TRIES = 100_000

comparison_rows = []
for problem_name, csp in PROBLEMS.items():
    for method_name, options in METHODS.items():
        # TODO: จับเวลา เรียก backtracking_search(csp, max_tries=MAX_TRIES, **options)
        #       แล้วเพิ่มแถวผลลัพธ์ลงใน comparison_rows
        pass

comparison = pd.DataFrame(comparison_rows)
display(comparison)

In [ ]:
# เซลล์แสดงผลข้อ 3 (ไม่ต้องแก้): ตารางสรุปจำนวนการลองค่า
if len(comparison) == 0:
    print("ข้อ 3: ยังไม่ได้ทำ")
else:
    tries_table = comparison.assign(tries=[f"{t:,}" if done else f"> {t:,} (ไม่เสร็จ)"
                                           for t, done in zip(comparison["tries"], comparison["เสร็จ"])])
    display(tries_table.pivot(index="เทคนิค", columns="ปัญหา", values="tries").reindex(list(METHODS)))
    finished = comparison[comparison["เสร็จ"]]
    print("คำตอบทุกชุดที่หาเสร็จถูกต้อง:", bool(finished["ถูกต้อง"].all()))

### ข้อ 4 วิเคราะห์ผล

1. เทคนิคใดลดจำนวนการลองค่าของซูโดกุได้มากที่สุดเมื่อใช้อย่างเดียว MRV หรือ FC เพราะอะไร และเมื่อใช้ร่วมกันได้ผลอย่างไร
2. สำหรับปัญหาซ่อมบำรุง ถ้าไม่ทำ AC-3 ก่อน (ใช้โดเมนเต็ม) MRV + FC + LCV ยังย้อนรอยหรือไม่ เทียบกับผลในเซลล์ตรวจข้อ 1–2
3. ในตาราง LCV ช่วยลดจำนวนการลองค่าได้ทุกปัญหาหรือไม่ แต่ละครั้งที่เรียก LCV มีต้นทุนอะไรเพิ่มขึ้น (ดูคอลัมน์เวลา)
4. ถ้าต้องจัดตารางซ่อมบำรุงอุปกรณ์ 200 รายการใน 30 วัน ควรเลือกใช้เทคนิคใด เพราะอะไร

*คำตอบ:* (เขียนคำตอบที่นี่)

## สรุป

- **Minimax** คำนวณค่าย้อนจากใบไม้ขึ้นมาราก โดยสมมติว่าคู่แข่งเล่นดีที่สุด ในตัวอย่างการเสนอราคา A ควรเลือก $a_1$ ซึ่งรับประกันกำไรอย่างน้อย 3 (ส่วนที่ 1)
- **Alpha-Beta Pruning** ได้ค่าที่รากเท่า Minimax เสมอ แต่ประเมินใบไม้น้อยลง (7 จาก 9 ใบในตัวอย่างที่ 2.25 และเหลือ 5 ใบเมื่อเรียงลำดับดี) ในเกม OX จากกระดานว่าง Alpha-Beta เยี่ยมโหนดเพียงไม่กี่เปอร์เซ็นต์ของ 549,946 โหนดที่ Minimax เยี่ยม (ส่วนที่ 2–3)
- **ฟังก์ชันประเมิน** ทำให้ค้นหาแบบจำกัดความลึกได้ AI ที่มองเพียง 2 ตาด้วย EVAL ของตัวอย่างที่ 2.24 เล่นได้ดีพอสมควรโดยใช้โหนดน้อยมาก (ส่วนที่ 3)
- **CSP** นิยามด้วยตัวแปร โดเมน และข้อจำกัด การจัดตารางซ่อมบำรุงสถานีไฟฟ้าย่อยมี 3 คำตอบจาก 81 แบบ Backtracking พื้นฐานพบคำตอบแรกด้วยการลองค่า 15 ครั้งและย้อนรอย 1 ครั้ง ตรงกับตัวอย่างที่ 2.27 (ส่วนที่ 4)
- **MRV, Forward Checking, LCV และ AC-3** ลดการลองค่าและการย้อนรอยได้มาก การจัดตารางซ่อมบำรุงเหลือการกำหนดค่าเพียง 4 ครั้งโดยไม่ย้อนรอยตามตัวอย่างที่ 2.28 และซูโดกุยากแก้ได้ในเวลาสั้น (ส่วนที่ 5 และแบบฝึกปฏิบัติ)

**ก้าวต่อไป:** บทที่ 3 จะศึกษาการแทนความรู้และการให้เหตุผล ซึ่งเป็นอีกวิธีหนึ่งที่ช่วยให้เอเจนต์ตัดสินใจได้โดยใช้ความรู้ที่มีโครงสร้าง

## เอกสารอ้างอิง

- Russell, S., & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson. บทที่ 5 (Adversarial Search) และบทที่ 6 (Constraint Satisfaction Problems)
- Knuth, D. E., & Moore, R. W. (1975). An analysis of alpha-beta pruning. *Artificial Intelligence, 6*(4), 293–326.
- Mackworth, A. K. (1977). Consistency in networks of relations. *Artificial Intelligence, 8*(1), 99–118.
- Norvig, P. (n.d.). *Solving Every Sudoku Puzzle*. https://norvig.com/sudoku.html (ที่มาของ `HARD_PUZZLE`)
- Wikipedia. *Sudoku*. https://en.wikipedia.org/wiki/Sudoku (ที่มาของ `EASY_PUZZLE`)